# 🏭 Warehouse Multi-Agent System with LangGraph

## Building a Hierarchical Agent Workflow for Warehouse Operations 🤖

This notebook demonstrates how to build a **multi-agent system** using **LangGraph** to simulate a warehouse automation pipeline. You'll see how multiple specialized agents coordinate through a hierarchical workflow to fulfill orders.

### 🎯 What You'll Learn

By the end of this notebook, you'll understand:

1. **🏗️ Agent Class Design** - How to define specialized agent classes (Pickers, Transporters, Supervisors)
2. **🔀 Hierarchical Delegation** - How a top-level supervisor delegates to zone supervisors, who delegate to workers
3. **⚡ LangGraph StateGraph** - How to wire agents together using nodes, edges, and conditional routing
4. **🧪 Simulation Execution** - How to stream through the workflow and observe task completion step-by-step

### 🌟 Why Multi-Agent Systems?

Real-world warehouses use **hierarchical coordination** — managers assign orders to zones, zone leads assign picks, and transport robots move goods. This notebook mirrors that pattern:

- **TopLevelSupervisor** → receives an order and routes it to the correct zone
- **ZoneSupervisor** → delegates the task to a picker robot in its zone
- **PickerRobot** → picks the items from shelves
- **TransportRobot** → moves picked items to the shipping area

### 📋 Prerequisites

- **LangGraph** installed (`langgraph` package)
- Basic understanding of Python classes and dictionaries
- Familiarity with graph-based workflow concepts

---

*💡 **Pro Tip**: This demo uses simple in-memory state — no LLM calls needed! It focuses purely on the multi-agent orchestration pattern.*

## 1. 🔧 Environment Setup & Imports

### 📦 Package Overview

We only need two core LangGraph components for this demo:

- **`StateGraph`** - The main class for building our multi-agent workflow as a directed graph
- **`END`** - A special sentinel node that marks the termination point of the workflow
- **`MemorySaver`** - An in-memory checkpointer that enables state persistence and replay

### 🔑 Key Concept: StateGraph
A `StateGraph` lets you define **nodes** (agent processing steps) and **edges** (transitions between agents). Each node receives the current state dictionary and returns an updated version.

In [ ]:
# ============================================================
# 📦 Core LangGraph Imports
# ============================================================
# StateGraph: The backbone class that lets us define a workflow
#             as a graph of interconnected agent nodes
# END:        A special terminal node — when an agent transitions
#             to END, the workflow stops and returns the final state
# MemorySaver: An in-memory checkpointer that saves workflow state
#              at each step, enabling pause/resume and replay
# ============================================================
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

print("✅ All imports loaded successfully!")

In [ ]:
# (Optional: Add custom source directories to Python path if needed)
# sys.path.insert(0, os.path.join(os.path.dirname(__file__), 'src'))

## 2. 🤖 Define the Agent Classes

### 🏗️ Agent Hierarchy Architecture

Our warehouse system uses four types of agents organized in a **command hierarchy**:

```
    TopLevelSupervisor          ← Receives orders, routes to zone
           │
      ZoneSupervisor            ← Manages a zone, delegates to workers
           │
      PickerRobot               ← Picks items from shelves
           │
     TransportRobot             ← Moves picked items to shipping
```

### 🔑 Key Design Principles:

- **Each agent has a `process()` method** that takes the current state dict and returns an updated state
- **Supervisors set `state['next_agent']`** to control which agent runs next (conditional routing)
- **Workers update `state['active_tasks']`** to record completed work
- **Agent metadata** (`agent_id`, `agent_type`, `capabilities`) enables introspection and logging

### 💡 Why This Pattern?
This mirrors real-world warehouse automation where a central controller dispatches orders to zone managers, who then coordinate individual robots. Each layer only needs to know about the layer directly below it.

In [ ]:
# ============================================================
# 🤖 PickerRobot — Worker agent that picks items from shelves
# ============================================================
# This is the lowest-level worker in our hierarchy. It simulates
# a physical robot that retrieves items from warehouse shelves.
# ============================================================
class PickerRobot:
    def __init__(self, robot_id, zone_id):
        # Unique identifier combining role prefix with robot ID (e.g., "picker_AP1")
        self.agent_id = f"picker_{robot_id}"
        # The warehouse zone this robot is assigned to (e.g., "A")
        self.zone_id = zone_id
        # Agent type used for hierarchy classification
        self.agent_type = "worker"
        # List of capabilities this robot can perform
        self.capabilities = ["item_picking"]

    def process(self, state):
        """Process function called by LangGraph when this node executes.
        Takes the current state dict and returns the updated state."""
        print(f"[Picker {self.agent_id}] Picking items in zone {self.zone_id}")
        # Record this robot's task completion in the shared state
        state['active_tasks'][self.agent_id] = 'picked'
        return state


# ============================================================
# 🚚 TransportRobot — Worker agent that moves items to shipping
# ============================================================
# After a PickerRobot picks items, a TransportRobot carries them
# to the shipping area. This is the final worker step.
# ============================================================
class TransportRobot:
    def __init__(self, robot_id, zone_id):
        self.agent_id = f"transport_{robot_id}"
        self.zone_id = zone_id
        self.agent_type = "worker"
        self.capabilities = ["item_transport"]

    def process(self, state):
        """Transport the picked items — updates state with 'transported' status."""
        print(
            f"[Transport {self.agent_id}] Transporting items in zone {self.zone_id}")
        # Mark transport as complete in shared state
        state['active_tasks'][self.agent_id] = 'transported'
        return state


# ============================================================
# 👷 ZoneSupervisor — Mid-level supervisor managing one zone
# ============================================================
# Each warehouse zone has a supervisor that receives tasks from
# the top-level supervisor and delegates to worker robots.
# The key behavior: it sets state['next_agent'] to route to
# the correct worker — this drives LangGraph's conditional edges.
# ============================================================
class ZoneSupervisor:
    def __init__(self, zone_id):
        self.agent_id = f"zone_sup_{zone_id}"
        self.zone_id = zone_id
        self.agent_type = "supervisor"

    def process(self, state):
        """Delegate work to the next available worker in this zone."""
        print(f"[ZoneSupervisor {self.zone_id}] Delegating tasks to workers")
        # Determine which worker should handle the task next
        # Defaults to the picker robot for this zone (e.g., "picker_AP1")
        next_worker = state.get('next_worker', f"picker_{self.zone_id}P1")
        # Set the routing key that LangGraph's conditional edge will read
        state['next_agent'] = next_worker
        return state


# ============================================================
# 🏢 TopLevelSupervisor — Top of the hierarchy, receives orders
# ============================================================
# This is the entry point of the workflow. It receives incoming
# orders and routes them to the appropriate zone supervisor.
# In a production system, it would analyze the order to determine
# which zone contains the requested items.
# ============================================================
class TopLevelSupervisor:
    def __init__(self):
        self.agent_id = "top_supervisor_main"
        self.agent_type = "supervisor"

    def process(self, state):
        """Route incoming orders to the correct zone supervisor."""
        print("[TopSupervisor] Assigning order to zone supervisor")
        # For this demo, all orders route to Zone A's supervisor
        # In production, this would analyze item locations to pick the right zone
        state['next_agent'] = 'zone_sup_A'
        return state

print("✅ All agent classes defined successfully!")

## 3. ⚡ Initial State & Workflow Configuration

### 🧠 Understanding the Shared State

In LangGraph, **all agents share a single state dictionary** that flows through the graph. Each agent reads from it, modifies it, and passes it along. Our state contains:

| Key | Purpose |
|---|---|
| `thread_id` | Unique identifier for this workflow execution (used by the checkpointer) |
| `pending_orders` | List of orders to be fulfilled — each with items and quantities |
| `active_tasks` | Dictionary tracking which agents have completed their work |
| `next_agent` | Routing key — supervisors set this to control which node runs next |
| `next_worker` | Hint for the zone supervisor about which worker to activate |

### 💡 Why a Shared State?
This pattern enables **loose coupling** between agents — they don't call each other directly. Instead, they communicate by reading and writing to the shared state, and LangGraph handles the routing.

In [ ]:
# ============================================================
# 📋 Create the initial state for our warehouse simulation
# ============================================================
# This function builds the starting state dictionary that will
# flow through every agent in the workflow graph.
# ============================================================
def create_initial_state():
    return {
        # Thread ID for the checkpointer — enables state persistence & replay
        'thread_id': 'demo-001',
        # The order(s) to fulfill — contains item IDs and quantities
        'pending_orders': [{'order_id': 'ORD_001', 'items': [{'item_id': 'BOOK_001', 'quantity': 2}]}],
        # Tracks completed work by each agent (starts empty, fills as agents process)
        'active_tasks': {},
        # The first agent to execute — the top-level supervisor starts the chain
        'next_agent': 'top_supervisor_main',
        # Hint for the zone supervisor: which picker robot to activate
        'next_worker': 'picker_AP1',
    }

print("✅ Initial state factory defined!")

## 4. 🏗️ Step 1: Instantiate the Agents

### 🔧 Creating Our Agent Team

Now we create concrete instances of each agent class and store them in a dictionary. The dictionary keys become the **node names** in our LangGraph workflow — this is how the graph knows which agent to call.

### 📊 Our Agent Roster:
| Agent ID | Class | Role |
|---|---|---|
| `top_supervisor_main` | TopLevelSupervisor | Routes orders to zones |
| `zone_sup_A` | ZoneSupervisor | Manages Zone A workers |
| `picker_AP1` | PickerRobot | Picks items in Zone A |
| `transport_AT1` | TransportRobot | Transports items in Zone A |

In [ ]:
# ============================================================
# 🤖 Instantiate all agents and register them by ID
# ============================================================
# The dictionary keys here will become node names in the graph.
# Each value is an agent instance whose .process() method
# will be called when that node executes.
# ============================================================
agents = {
    'top_supervisor_main': TopLevelSupervisor(),       # 🏢 Top-level: routes orders to zones
    'zone_sup_A': ZoneSupervisor('A'),                 # 👷 Zone A supervisor: delegates to workers
    'picker_AP1': PickerRobot('AP1', 'A'),             # 🤖 Picker robot in Zone A
    'transport_AT1': TransportRobot('AT1', 'A'),       # 🚚 Transport robot in Zone A
}

print(f"✅ Created {len(agents)} agents: {list(agents.keys())}")

## 5. 🔀 Step 2: Build the LangGraph Workflow

### 🧠 Wiring Agents into a Graph

This is where the magic happens! We create a `StateGraph` and connect our agents using three types of connections:

### 🔗 Edge Types Explained:

| Edge Type | Description | Used For |
|---|---|---|
| **`add_node()`** | Registers an agent's process function as a graph node | Every agent |
| **`add_conditional_edges()`** | Routes to different nodes based on a runtime condition | Supervisors (dynamic routing) |
| **`add_edge()`** | Creates a fixed, unconditional transition between nodes | Worker-to-worker handoffs |
| **`set_entry_point()`** | Designates which node runs first | The top supervisor |

### ⚡ Workflow Flow:
```
[Entry] → TopSupervisor ---(conditional)--→ ZoneSupervisor_A
                                                    │
                                            (conditional)
                                                    │
                                              PickerRobot_AP1
                                                    │
                                              (fixed edge)
                                                    │
                                            TransportRobot_AT1
                                                    │
                                                  [END]
```

### 💡 Key Insight: Conditional vs. Fixed Edges
**Conditional edges** read a value from the state (like `state['next_agent']`) to decide where to go — this enables dynamic, data-driven routing. **Fixed edges** always go to the same next node.

In [ ]:
# ============================================================
# 🔀 Build the LangGraph workflow from our agent instances
# ============================================================

# Create a StateGraph using a plain dict as the state schema
# (LangGraph also supports TypedDict for stricter typing)
workflow = StateGraph(dict)

# --- Step A: Register each agent as a node in the graph ---
# The node name (agent_id) maps to the agent's .process() method
for agent_id, agent in agents.items():
    workflow.add_node(agent_id, agent.process)

# --- Step B: Add CONDITIONAL edges for supervisors ---
# The top supervisor reads state['next_agent'] to decide routing
# The lambda function extracts the routing key from state
# The dict maps possible routing values to target node names
workflow.add_conditional_edges(
    'top_supervisor_main',                    # Source node
    lambda state: state['next_agent'],        # Routing function (reads state)
    {'zone_sup_A': 'zone_sup_A'}             # Routing map: value → target node
)

# Zone supervisor also uses conditional routing to pick a worker
workflow.add_conditional_edges(
    'zone_sup_A',                             # Source node
    lambda state: state['next_agent'],        # Routing function
    {'picker_AP1': 'picker_AP1'}             # Routing map
)

# --- Step C: Add FIXED edges for worker-to-worker handoffs ---
# After picking, always transport — no decision needed
workflow.add_edge('picker_AP1', 'transport_AT1')

# After transport, the workflow is complete — transition to END
workflow.add_edge('transport_AT1', END)

# --- Step D: Set the entry point ---
# The top supervisor is always the first node to execute
workflow.set_entry_point('top_supervisor_main')

# --- Step E: Compile the graph with a checkpointer ---
# MemorySaver() enables state snapshots at each step for replay/debugging
agent_graph = workflow.compile(checkpointer=MemorySaver())

print("✅ Workflow graph compiled successfully!")

### 🎨 Visualize the Workflow Graph

LangGraph can render a visual diagram of the compiled workflow using Mermaid. This is an excellent way to verify that your edges and nodes are connected correctly before running the simulation.

In [ ]:
# ============================================================
# 🎨 Render the workflow graph as a visual diagram
# ============================================================
# get_graph() returns the graph structure, draw_mermaid_png()
# renders it as a PNG image using the Mermaid diagramming library
# ============================================================
from IPython.display import display, HTML, Image
display(Image(agent_graph.get_graph().draw_mermaid_png()))

## 6. 🧪 Step 3: Run the Warehouse Simulation

### 🚀 Executing the Workflow End-to-End

Now we'll run the full workflow! Here's what will happen at each step:

1. **TopSupervisor** receives the order and sets `next_agent = 'zone_sup_A'`
2. **ZoneSupervisor A** receives the delegation and sets `next_agent = 'picker_AP1'`
3. **PickerRobot AP1** picks the items and records `'picked'` in `active_tasks`
4. **TransportRobot AT1** transports items and records `'transported'` in `active_tasks`
5. Workflow reaches **END** — simulation complete!

### 🔑 Key Concepts in This Cell:
- **`agent_graph.stream()`** — Yields the state after *each* node executes (great for debugging)
- **`stream_mode="values"`** — Returns the full state dict at each step (vs. just deltas)
- **`config`** — The `thread_id` tells the checkpointer which execution thread to track

In [ ]:
# ============================================================
# 🚀 Execute the warehouse simulation end-to-end
# ============================================================
print("\n🚀 Starting simple warehouse simulation...")

# Create the initial state with our order and routing defaults
initial_state = create_initial_state()

# Stream through the workflow, printing state at each step
# stream() yields the state after each node processes it
# stream_mode="values" returns the complete state dict (not just changes)
for step, result in enumerate(
    agent_graph.stream(
        initial_state,
        stream_mode="values",
        # Config tells the checkpointer which thread to track
        # This enables state persistence, replay, and debugging
        config={"configurable": {"thread_id": initial_state["thread_id"]}}
    )
):
    # Print the active_tasks dict to show which agents have completed work
    print(f"  📦 Step {step+1}: active_tasks = {result['active_tasks']}")

print("\n✅ Simple warehouse workflow complete!")

## 📝 Summary & Key Takeaways

### What We Built
We created a **hierarchical multi-agent warehouse system** using LangGraph's `StateGraph`, demonstrating how multiple agents coordinate through a shared state dictionary.

### 🔑 Core Concepts Reviewed

| Concept | What We Learned |
|---|---|
| **Agent Classes** | Each agent has a `process(state)` method that reads and updates shared state |
| **StateGraph** | A directed graph where nodes are agents and edges are transitions |
| **Conditional Edges** | Supervisors dynamically route to the next agent based on state values |
| **Fixed Edges** | Worker-to-worker handoffs use unconditional transitions |
| **Checkpointing** | `MemorySaver` records state at each step for debugging and replay |
| **Streaming** | `stream()` lets you observe the workflow step-by-step |

### 🚀 Next Steps
- **Scale up**: Add more zones (B, C) with their own supervisors and workers
- **Add LLM reasoning**: Replace simple routing logic with LLM-based decision making
- **Error handling**: Add retry logic and fallback paths for failed tasks
- **Explore**: See the **Supervisor-Agent** and **Swarm-Agent** notebooks for LLM-powered multi-agent patterns!